# 🔬 Fluid Bed Pellet & Agglomeration Simulation Workbench
### High-Fidelity Synthetic Dataset Generation for Detecting Agglomeration in Fluid Bed Process Imaging

---

## 📌 Project Overview
In pharmaceutical fluid-bed coating and granulation, detecting **agglomeration** (unintended fusing/sticking of 2, 3, 4, or more pellets) is critical for quality control.
Manual data annotation is extremely laborious and time-consuming because particles move rapidly in a turbulent air stream and can appear blurred or partially occluded.

This workbench provides a **lightweight, physics-based, photorealistic 3D simulation engine** specifically calibrated to the real camera sensor in `/Real-Data/T11/B4`:
1. **Accurate Photometric Shading**: Directional coaxial illumination, Lambertian diffuse, Blinn-Phong specular highlights, and micro-roughness.
2. **Organic Agglomerate Morphology**: Mathematical smooth-minimum (`smin`) liquid-bridge meniscus necking between 2, 3, 4, or more connected pellets.
3. **Optics & Sensor Effects**: Depth of Field (defocus blur as a function of depth $z$), velocity-aligned motion blur, and Poisson-Gaussian camera sensor noise.
4. **Fluid Bed Kinematics**: Free 3D particle motion (upward central spout flow, downward wall recirculation, and 3D tumbling rotation).
5. **Automated ML Ground-Truth Export**: YOLO detection (`yolo.txt`), YOLO instance segmentation polygons, LabelMe JSON (`shapes`), and temporal tracking logs.


### 1. Environment & Package Imports

In [ ]:
import os
import glob
import json
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Import our custom simulation package
from simulation import (
    SimulationConfig,
    PelletEntity,
    PelletType,
    OpticalRenderer,
    BackgroundEngine,
    FluidBedKinematics,
    DatasetExporter
)

print("Simulation package successfully loaded!")


### 2. Inspecting Real Fluid Bed Data (`/Real-Data/T11/B4`)

In [ ]:
real_files = sorted(glob.glob('Real-Data/T11/B4/*.bmp'))
print(f"Found {len(real_files)} real images in Real-Data/T11/B4.")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i in range(min(3, len(real_files))):
    img = cv2.imread(real_files[i], cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Real Frame {i}: {os.path.basename(real_files[i])}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


### 3. Static Chamber Background Extraction
By computing the **temporal median** across frames, we eliminate fast-moving foreground particles, isolating the true chamber geometry (metallic nozzle on the right, bottom rim, and background bokeh haze).


In [ ]:
config = SimulationConfig()
bg_engine = BackgroundEngine(config)

# Extract and cache background plate
bg_plate = bg_engine.extract_from_real_data('Real-Data/T11/B4')
print(f"Extracted Background Plate shape: {bg_plate.shape}")

plt.figure(figsize=(10, 7))
plt.imshow(bg_plate, cmap='gray')
plt.title("Extracted Chamber Background Plate (Temporal Median)")
plt.axis('off')
plt.show()


### 4. Interactive 3D Pellet Morphology & Optical Rendering
Here we test rendering:
- **Single Pellet** ($N=1$)
- **Dimer Agglomerate** ($N=2$, connected dumbbell)
- **Trimer Agglomerate** ($N=3$, chain or triangular cluster)
- **Tetramer & Cluster** ($N=4+$, multi-pellet agglomerate)


In [ ]:
renderer = OpticalRenderer(config)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
test_cases = [
    ("Single Pellet (N=1)", 1),
    ("Dimer Agglomerate (N=2)", 2),
    ("Trimer Agglomerate (N=3)", 3),
    ("Cluster Agglomerate (N=5)", 5)
]

for idx, (title, count) in enumerate(test_cases):
    entity = PelletEntity(entity_id=idx+1, count=count, base_radius=config.base_radius)
    entity.velocity = np.array([random.uniform(-3, 3), random.uniform(-6, -2), 0.0])
    
    fg, alpha, lbox, lcontour, _ = renderer.render_entity_patch(entity)
    
    # Display patch with synthetic background
    patch_vis = fg + 35.0 * (1.0 - alpha)
    ax = axes[idx // 2, idx % 2]
    ax.imshow(patch_vis, cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()


### 5. Domain Alignment: Real Crops vs Synthetic Renders
Let's directly compare a real single pellet and real agglomerate against our simulated renders.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

demo_dir = 'outputs/demo'

# 1. Real single pellet
real_single_path = os.path.join(demo_dir, 'crop_pellet_1.png')
if os.path.exists(real_single_path):
    real_single = cv2.imread(real_single_path, cv2.IMREAD_GRAYSCALE)
else:
    b4 = cv2.imread('Real-Data/T11/B4/Pic_20261011144014002-92.bmp', cv2.IMREAD_GRAYSCALE)
    real_single = b4[1025:1062, 1304:1342] if b4 is not None else np.zeros((37, 37), np.uint8)

# 2. Simulated single pellet
synth_single_path = os.path.join(demo_dir, 'synth_pellet_test.png')
if os.path.exists(synth_single_path):
    synth_single = cv2.imread(synth_single_path, cv2.IMREAD_GRAYSCALE)
else:
    ent1 = PelletEntity(entity_id=1, count=1, base_radius=config.base_radius)
    fg1, a1, _, _, _ = renderer.render_entity_patch(ent1)
    synth_single = (fg1 + 30.0 * (1.0 - a1)).astype(np.uint8)

# 3. Real trimer agglomerate
real_trimer_path = os.path.join(demo_dir, 'b6_crop_1.png')
if os.path.exists(real_trimer_path):
    real_trimer = cv2.imread(real_trimer_path, cv2.IMREAD_GRAYSCALE)
else:
    b6 = cv2.imread('Real-Data/T11/B6/Pic_20261011144620761-121.bmp', cv2.IMREAD_GRAYSCALE)
    real_trimer = b6[923:1010, 1004:1056] if b6 is not None else np.zeros((87, 52), np.uint8)

# 4. Simulated trimer agglomerate
synth_trimer_path = os.path.join(demo_dir, 'synth_trimer_photoreal.png')
if os.path.exists(synth_trimer_path):
    synth_trimer = cv2.imread(synth_trimer_path, cv2.IMREAD_GRAYSCALE)
else:
    ent3 = PelletEntity(entity_id=2, count=3, base_radius=config.base_radius)
    fg3, a3, _, _, _ = renderer.render_entity_patch(ent3)
    synth_trimer = (fg3 + 30.0 * (1.0 - a3)).astype(np.uint8)

axes[0, 0].imshow(real_single, cmap='gray')
axes[0, 0].set_title('Real Single Pellet', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(synth_single, cmap='gray')
axes[0, 1].set_title('Simulated Single Pellet', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(real_trimer, cmap='gray')
axes[1, 0].set_title('Real Trimer Agglomerate (B6)', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(synth_trimer, cmap='gray')
axes[1, 1].set_title('Simulated Trimer Agglomerate', fontsize=11, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

### 6. Dynamic Fluid Bed Swarm Kinematics
The particles move freely according to fluid-bed aerodynamics:
- Upward fountain core
- Downward outer wall recirculation
- Continuous 3D tumbling rotation


In [ ]:
kinematics = FluidBedKinematics(config)

# Step simulation 3 times and visualize particle positions
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for step_i in range(3):
    kinematics.step(dt=1.0)
    bg = bg_engine.get_background(jitter_brightness=True)
    frame_img, annotations = renderer.render_scene(bg, kinematics.entities)
    
    # Draw annotations
    annotated = cv2.cvtColor(frame_img, cv2.COLOR_GRAY2RGB)
    for a in annotations:
        x1, y1, x2, y2 = a['bbox_xyxy']
        col = (0, 255, 0) if a['count'] == 1 else (255, 100, 0)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), col, 2)
        cv2.putText(annotated, f"N={a['count']}", (x1, max(18, y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
        
    axes[step_i].imshow(annotated)
    axes[step_i].set_title(f"Simulation Frame {step_i+1} ({len(annotations)} In-Focus Objects)")
    axes[step_i].axis('off')

plt.tight_layout()
plt.show()


### 7. Generate Full ML Training Dataset
Run the cell below to generate a synthetic dataset with YOLO format and LabelMe JSON ground-truth.


In [ ]:
output_dataset_dir = "outputs/synthetic_pellet_dataset"
num_frames_to_generate = 50

exporter = DatasetExporter(config, output_dir=output_dataset_dir)
exporter.setup_directories()

print(f"Generating {num_frames_to_generate} frames...")
for f_idx in range(num_frames_to_generate):
    kinematics.step(dt=1.0)
    bg = bg_engine.get_background(jitter_brightness=True)
    frame_img, annotations = renderer.render_scene(bg, kinematics.entities)
    
    split = 'train' if f_idx < int(num_frames_to_generate * 0.8) else 'val'
    exporter.save_frame(
        frame_idx=f_idx,
        image=frame_img,
        annotations=annotations,
        split=split,
        export_labelme=True,
        export_yolo_seg=True
    )

print(f"Generated {num_frames_to_generate} frames in {output_dataset_dir}!")
print(f"YAML config created at: {os.path.join(output_dataset_dir, 'dataset.yaml')}")


### 8. Training YOLO on Synthetic Data & Transferring to Real Data
You can now train a YOLO detector or segmenter directly using Ultralytics:

```bash
# In terminal:
python -m pip install ultralytics
yolo detect train data=synthetic_pellet_dataset/dataset.yaml model=yolo11n.pt epochs=25 imgsz=1024 batch=4 device=0
```

Or run directly in Python:
```python
from ultralytics import YOLO

model = YOLO("yolo11n.pt") # Or yolo11n-seg.pt for instance segmentation
results = model.train(
    data="synthetic_pellet_dataset/dataset.yaml",
    epochs=25,
    imgsz=1024,
    device=0
)

# Test on real images in Real-Data/T11/B4!
val_results = model.predict(source="Real-Data/T11/B4/*.bmp", conf=0.35, save=True)
```
